In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# CELL 2 — Install Stable Roop Environment (BEST VERSION)

!git clone https://github.com/s0md3v/roop.git

!pip install -q customtkinter
!pip install -q insightface==0.7.3
!pip install -q onnxruntime-gpu
!pip install -q opencv-python pillow tqdm
!pip install -q gfpgan basicsr facexlib

fatal: destination path 'roop' already exists and is not an empty directory.


In [3]:
# CELL 3 — Confirm GPU is Ready

import onnxruntime as ort
print(ort.get_available_providers())

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [4]:
# CELL 4 — Check Dataset Files in Drive

import glob

videos = sorted(glob.glob("/content/drive/MyDrive/deepfake_v2/roop_input_v2/*.mp4"))
donors = sorted(glob.glob("/content/drive/MyDrive/deepfake_v2/donor_faces/*.*"))

print("Videos found :", len(videos))
print("Donors found :", len(donors))

Videos found : 92
Donors found : 10


In [5]:
!pip install -q tkinterdnd2
!pip install -q opennsfw2

In [6]:
# CELL 5 — TEST FULL FAKE GENERATION

import os
import subprocess
import random

out_dir = "/content/drive/MyDrive/deepfake_v2/fake_full_v2"
os.makedirs(out_dir, exist_ok=True)

random.seed(42)

done = 0
failed = []

for v in videos:

    vid = os.path.basename(v).replace(".mp4", "")
    donor = random.choice(donors)
    output = f"{out_dir}/{vid}_fake.mp4"

    if os.path.exists(output) and os.path.getsize(output) > 500000:
        print("Already done:", vid)
        done += 1
        continue

    print("\nProcessing:", vid)

    result = subprocess.run(
        [
            "python", "roop/run.py",
            "-s", donor,
            "-t", v,
            "-o", output,
            "--many-faces",
            "--skip-audio",
            "--execution-provider", "cuda"
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr)

    if os.path.exists(output) and os.path.getsize(output) > 500000:
        print("SUCCESS:", vid)
        done += 1
    else:
        print("FAILED:", vid)
        failed.append(vid)

print("\nDONE:", done)
print("FAILED:", failed)

Streaming output truncated to the last 5000 lines.
Processing: 100%|██████████| 3600/3600 [07:01<00:00,  8.55frame/s, memory_usage=02.68GB, execution_providers=['CUDAExecutionProvider'], execution_threads=8]

SUCCESS: tamil_034

DONE: 92
FAILED: []
